In [8]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [9]:
spark = SparkSession.builder \
    .appName("ComprehensiveHospitalAnalytics") \
    .master("local[*]") \
    .getOrCreate()

In [10]:
os.makedirs("data", exist_ok=True)

In [11]:
with open("data/patients.csv", "w") as f:
    f.write("""patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active""")

In [12]:
with open("data/appointments.csv", "w") as f:
    f.write("""appointment_id,patient_id,doctor_name,department,appointment_date,consultation_fee,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2025-01-22,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending""")

In [13]:
with open("data/patient_preferences.json", "w") as f:
    f.write("""[
{"patient_id":101,"preferred_hospital":"Apollo","contact":{"phone":"9876500011","email":"rahul@gmail.com"}},
{"patient_id":102,"preferred_hospital":"Yashoda","contact":{"phone":null,"email":"priya@gmail.com"}},
{"patient_id":103,"preferred_hospital":"Care","contact":{"phone":"9876500013","email":null}},
{"patient_id":104,"preferred_hospital":null,"contact":{"phone":"9876500014","email":"sneha@gmail.com"}}
]""")

In [14]:
print("\n--- Part 1: CSV Ingestion ---")


--- Part 1: CSV Ingestion ---


In [15]:
patients_df = spark.read.csv("data/patients.csv", header=True, inferSchema=True)
appointments_df = spark.read.csv("data/appointments.csv", header=True, inferSchema=True)

In [16]:
print("Patients Schema:")
patients_df.printSchema()
print("Appointments Schema:")
appointments_df.printSchema()

Patients Schema:
root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

Appointments Schema:
root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- status: string (nullable = true)



In [17]:
print(f"Patients Record Count: {patients_df.count()}")
print(f"Appointments Record Count: {appointments_df.count()}")

Patients Record Count: 10
Appointments Record Count: 10


In [18]:
patients_df.show(5)
appointments_df.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|   

In [19]:
print("Distinct Cities:")
patients_df.select("city").distinct().show()

Distinct Cities:
+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [20]:
print("Distinct Departments:")
appointments_df.select("department").distinct().show()

Distinct Departments:
+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [21]:
patients_df.write.mode("overwrite").parquet("data/patients.parquet")
parquet_patients_df = spark.read.parquet("data/patients.parquet")

In [22]:
print(f"CSV Count: {patients_df.count()} | Parquet Count: {parquet_patients_df.count()}")

CSV Count: 10 | Parquet Count: 10


In [23]:
print("\n--- Part 2: Filtering ---")


--- Part 2: Filtering ---


In [24]:
patients_df.filter(F.col("city") == "Hyderabad").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [25]:
patients_df.filter(F.col("gender") == "Female").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [26]:
patients_df.filter(F.col("age") > 40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [27]:
appointments_df.filter(F.col("status") == "Completed").show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+-

In [28]:
appointments_df.filter(F.col("status") == "Pending").show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [29]:
appointments_df.filter(F.col("consultation_fee") > 1500).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [30]:
patients_df.filter(F.col("insurance_status") == "Active").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [31]:
patients_df.filter(F.col("insurance_status") == "Inactive").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [32]:
patients_df.filter(F.col("blood_group") == "O+").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [33]:
appointments_df.filter(F.col("department") == "Cardiology").show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [34]:
print("\n--- Part 3: Null Handling ---")


--- Part 3: Null Handling ---


In [35]:
patients_df.filter(F.col("city").isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [36]:
patients_df.filter(F.col("blood_group").isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [37]:
appointments_df.filter(F.col("consultation_fee").isNull()).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [38]:
print("Nulls count in patients:")
patients_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in patients_df.columns]).show()

Nulls count in patients:
+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+



In [39]:
cleaned_patients = patients_df.na.fill({"city": "Unknown", "blood_group": "Not Available"})
cleaned_appointments = appointments_df.na.fill({"consultation_fee": 0})

In [40]:
appointments_df.na.drop(subset=["consultation_fee"]).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [41]:
patients_dq = patients_df.withColumn(
    "data_quality_status",
    F.when(F.col("city").isNull() | F.col("blood_group").isNull(), "Incomplete").otherwise("Complete")
)
patients_dq.show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|         Incomplete|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|           Complete|
|       108|  Meera Nair|    Kochi| 48|F

In [42]:
patients_dq.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    8|
|         Incomplete|    2|
+-------------------+-----+



In [43]:
print("\n--- Part 4: Built-in Functions ---")


--- Part 4: Built-in Functions ---


In [44]:
patients_df.withColumn("name_upper", F.upper(F.col("patient_name"))).show(3)
patients_df.withColumn("name_lower", F.lower(F.col("patient_name"))).show(3)

+----------+------------+---------+---+------+-----------+----------------+------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|  name_upper|
+----------+------------+---------+---+------+-----------+----------------+------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|RAHUL SHARMA|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active| PRIYA REDDY|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|  AMIT KUMAR|
+----------+------------+---------+---+------+-----------+----------------+------------+
only showing top 3 rows
+----------+------------+---------+---+------+-----------+----------------+------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|  name_lower|
+----------+------------+---------+---+------+-----------+----------------+------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|rahul sharm

In [45]:
patients_df.withColumn("name_len", F.length(F.col("patient_name"))).show(3)
patients_df.withColumn("name_prefix", F.substring(F.col("patient_name"), 1, 3)).show(3)

+----------+------------+---------+---+------+-----------+----------------+--------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|name_len|
+----------+------------+---------+---+------+-----------+----------------+--------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|      12|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|      11|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|      10|
+----------+------------+---------+---+------+-----------+----------------+--------+
only showing top 3 rows
+----------+------------+---------+---+------+-----------+----------------+-----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|name_prefix|
+----------+------------+---------+---+------+-----------+----------------+-----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|        Rah|
|       102| Priya Reddy|Bang

In [46]:
patients_df.withColumn("age_group",
    F.when(F.col("age") < 30, "Young")
     .when((F.col("age") >= 30) & (F.col("age") <= 50), "Middle-aged")
     .otherwise("Senior")
).show(5)

+----------+------------+---------+---+------+-----------+----------------+-----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|  age_group|
+----------+------------+---------+---+------+-----------+----------------+-----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|Middle-aged|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|      Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|Middle-aged|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|Middle-aged|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|     Senior|
+----------+------------+---------+---+------+-----------+----------------+-----------+
only showing top 5 rows


In [47]:
patients_df.withColumn("insurance_flag", F.when(F.col("insurance_status") == "Active", 1).otherwise(0)) \
           .withColumn("senior_citizen", F.when(F.col("age") >= 50, "Y").otherwise("N")).show(5)

+----------+------------+---------+---+------+-----------+----------------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-----------+----------------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|             1|             N|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|             1|             N|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|             0|             N|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|             1|             N|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|             1|             Y|
+----------+------------+---------+---+------+-----------+----------------+--------------+--------------+
only showing top 5 rows


In [48]:
patients_df.withColumn("name_city", F.concat_ws(" - ", F.col("patient_name"), F.col("city"))) \
           .withColumn("trimmed_name", F.trim(F.col("patient_name"))) \
           .withColumn("city_upper", F.upper(F.col("city"))).show(3)

+----------+------------+---------+---+------+-----------+----------------+--------------------+------------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|           name_city|trimmed_name|city_upper|
+----------+------------+---------+---+------+-----------+----------------+--------------------+------------+----------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|Rahul Sharma - Hy...|Rahul Sharma| HYDERABAD|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|Priya Reddy - Ban...| Priya Reddy| BANGALORE|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive| Amit Kumar - Mumbai|  Amit Kumar|    MUMBAI|
+----------+------------+---------+---+------+-----------+----------------+--------------------+------------+----------+
only showing top 3 rows


In [49]:
print("\n--- Part 5: GroupBy and Aggregations ---")


--- Part 5: GroupBy and Aggregations ---


In [50]:
cleaned_patients.groupBy("city").count().show()
cleaned_patients.groupBy("gender").count().show()
cleaned_patients.groupBy("blood_group").count().show()
appointments_df.groupBy("department").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    1|
|  Unknown|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+

+-------------+-----+
|  blood_group|count|
+-------------+-----+
|          AB+|    1|
|           O+|    2|
|           O-|    1|
|           B+|    2|
|           A+|    3|
|Not Available|    1|
+-------------+-----+

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [51]:
cleaned_patients.groupBy("city").agg(
    F.avg("age").alias("avg_age"),
    F.max("age").alias("max_age"),
    F.min("age").alias("min_age")
).show()

+---------+-------+-------+-------+
|     city|avg_age|max_age|min_age|
+---------+-------+-------+-------+
|Bangalore|   35.0|     41|     29|
|    Kochi|   48.0|     48|     48|
|  Chennai|   31.0|     31|     31|
|   Mumbai|   42.0|     42|     42|
|  Unknown|   38.0|     38|     38|
|     Pune|   26.0|     26|     26|
|    Delhi|   55.0|     55|     55|
|Hyderabad|   34.0|     35|     33|
+---------+-------+-------+-------+



In [52]:
dept_revenue = cleaned_appointments.groupBy("department").agg(
    F.avg("consultation_fee").alias("avg_fee"),
    F.sum("consultation_fee").alias("total_revenue")
)
dept_revenue.show()
print("Highest Revenue Generating Department:")
dept_revenue.orderBy(F.desc("total_revenue")).select("department").limit(1).show()

+-----------+-----------------+-------------+
| department|          avg_fee|total_revenue|
+-----------+-----------------+-------------+
|  Neurology|           2000.0|         4000|
|Dermatology|666.6666666666666|         2000|
| Cardiology|           1500.0|         4500|
|Orthopedics|           2500.0|         5000|
+-----------+-----------------+-------------+

Highest Revenue Generating Department:
+-----------+
| department|
+-----------+
|Orthopedics|
+-----------+



In [53]:
print("\n--- Part 6: Joins ---")


--- Part 6: Joins ---


In [54]:
patients_df.join(appointments_df, "patient_id", "inner").show(5)

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [55]:
patients_df.join(appointments_df, "patient_id", "left").show(5)

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [56]:
patients_df.join(appointments_df, "patient_id", "right").show(5)

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [57]:
patients_df.join(appointments_df, "patient_id", "full").show(5)

+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|          5002| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|       103|  Amit Kumar|   Mumbai| 42|  Male|

In [58]:
patients_df.join(appointments_df, "patient_id", "left_anti").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [59]:
appointments_df.join(patients_df, "patient_id", "left_anti").show()

+----------+--------------+-----------+----------+----------------+----------------+---------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|
+----------+--------------+-----------+----------+----------------+----------------+---------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+----------+--------------+-----------+----------+----------------+----------------+---------+



In [60]:
patient_summary = appointments_df.groupBy("patient_id").agg(
    F.count("appointment_id").alias("appointment_count"),
    F.sum("consultation_fee").alias("total_fees_paid")
)
patient_summary.show()

print("Highest Spending Patient ID:")
patient_summary.orderBy(F.desc("total_fees_paid")).select("patient_id").limit(1).show()

+----------+-----------------+---------------+
|patient_id|appointment_count|total_fees_paid|
+----------+-----------------+---------------+
|       108|                1|           NULL|
|       101|                2|           2500|
|       103|                1|           1500|
|       120|                1|           1500|
|       107|                1|           2000|
|       102|                1|           2000|
|       105|                1|           1000|
|       110|                1|           2500|
|       104|                1|           2500|
+----------+-----------------+---------------+

Highest Spending Patient ID:
+----------+
|patient_id|
+----------+
|       101|
+----------+



In [61]:
print("\n--- Part 7: Window Functions ---")


--- Part 7: Window Functions ---


In [62]:
win_fee_desc = Window.orderBy(F.desc("total_fees_paid"))
win_city_fee = Window.partitionBy("city").orderBy(F.desc("total_fees_paid"))
win_running_total = Window.orderBy("appointment_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
win_lead_lag = Window.orderBy("appointment_date")

patient_spend_profile = cleaned_patients.join(patient_summary, "patient_id", "inner")

patient_spend_profile.withColumn("rank", F.rank().over(win_fee_desc)) \
                     .withColumn("dense_rank", F.dense_rank().over(win_fee_desc)) \
                     .withColumn("row_number", F.row_number().over(win_fee_desc)).show()

+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+----+----------+----------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_count|total_fees_paid|rank|dense_rank|row_number|
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+----+----------+----------+
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|                1|           2500|   1|         1|         1|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|                2|           2500|   1|         1|         2|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|                1|           2500|   1|         1|         3|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|                1|           2000|   4|         2|         4|
|       107| Arjun Verma|     Pune| 26|  

In [63]:
ranked_spenders = patient_spend_profile.withColumn("rank", F.rank().over(win_fee_desc))
print("Top Patient:")
ranked_spenders.filter(F.col("rank") == 1).show()
print("Top 3 Patients:")
ranked_spenders.filter(F.col("rank") <= 3).show()

Top Patient:
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+----+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_count|total_fees_paid|rank|
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+----+
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|                1|           2500|   1|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|                2|           2500|   1|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|                1|           2500|   1|
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+----+

Top 3 Patients:
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+----+
|patient_id|patient_name|     city|age|gender|bloo

In [64]:
city_spending_rank = patient_spend_profile.withColumn("rank_desc", F.rank().over(win_city_fee)) \
                                          .withColumn("rank_asc", F.rank().over(Window.partitionBy("city").orderBy("total_fees_paid")))
print("Highest Spenders by City:")
city_spending_rank.filter(F.col("rank_desc") == 1).show()
print("Lowest Spenders by City:")
city_spending_rank.filter(F.col("rank_asc") == 1).show()

Highest Spenders by City:
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+---------+--------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|appointment_count|total_fees_paid|rank_desc|rank_asc|
+----------+------------+---------+---+------+-----------+----------------+-----------------+---------------+---------+--------+
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|                1|           2500|        1|       2|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|                1|           2500|        1|       1|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|                1|           1000|        1|       1|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|                2|           2500|        1|       1|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Act

In [65]:
cleaned_appointments.withColumn("running_total", F.sum("consultation_fee").over(win_running_total)) \
                    .withColumn("next_fee", F.lead("consultation_fee", 1).over(win_lead_lag)) \
                    .withColumn("prev_fee", F.lag("consultation_fee", 1).over(win_lead_lag)).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+-------------+--------+--------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|running_total|next_fee|prev_fee|
+--------------+----------+-----------+-----------+----------------+----------------+---------+-------------+--------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|         1500|    2000|    NULL|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|         3500|    1000|    1500|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|         4500|    1500|    2000|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|         6000|    2500|    1000|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|         

In [66]:
print("\n--- Part 8: JSON Processing ---")


--- Part 8: JSON Processing ---


In [67]:
pref_df = spark.read.option("multiline", "true").json("data/patient_preferences.json")
print("JSON Preference Schema:")
pref_df.printSchema()

JSON Preference Schema:
root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [68]:
flat_pref = pref_df.select(
    F.col("patient_id"),
    F.col("preferred_hospital"),
    F.col("contact.phone").alias("phone"),
    F.col("contact.email").alias("email")
)
flat_pref.show()

+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       101|            Apollo|9876500011|rahul@gmail.com|
|       102|           Yashoda|      NULL|priya@gmail.com|
|       103|              Care|9876500013|           NULL|
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [69]:
flat_pref.filter(F.col("phone").isNull()).show()
flat_pref.filter(F.col("email").isNull()).show()
flat_pref.filter(F.col("preferred_hospital").isNull()).show()

+----------+------------------+-----+---------------+
|patient_id|preferred_hospital|phone|          email|
+----------+------------------+-----+---------------+
|       102|           Yashoda| NULL|priya@gmail.com|
+----------+------------------+-----+---------------+

+----------+------------------+----------+-----+
|patient_id|preferred_hospital|     phone|email|
+----------+------------------+----------+-----+
|       103|              Care|9876500013| NULL|
+----------+------------------+----------+-----+

+----------+------------------+----------+---------------+
|patient_id|preferred_hospital|     phone|          email|
+----------+------------------+----------+---------------+
|       104|              NULL|9876500014|sneha@gmail.com|
+----------+------------------+----------+---------------+



In [70]:
cleaned_flat_pref = flat_pref.na.fill({"phone": "N/A", "email": "missing@hospital.com"})

In [71]:
patients_df.join(cleaned_flat_pref, "patient_id", "left").show()

+----------+------------+---------+---+------+-----------+----------------+------------------+----------+--------------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|preferred_hospital|     phone|               email|
+----------+------------+---------+---+------+-----------+----------------+------------------+----------+--------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|            Apollo|9876500011|     rahul@gmail.com|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Yashoda|       N/A|     priya@gmail.com|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|              Care|9876500013|missing@hospital.com|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|              NULL|9876500014|     sneha@gmail.com|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|              NULL|      NULL|      

In [72]:
print("\n--- Part 9: Spark SQL ---")


--- Part 9: Spark SQL ---


In [73]:
patients_df.createOrReplaceTempView("v_patients")
appointments_df.createOrReplaceTempView("v_appointments")

In [74]:
spark.sql("SELECT * FROM v_patients").show(3)
spark.sql("SELECT * FROM v_patients WHERE city = 'Hyderabad'").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 3 rows
+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+------

In [75]:
spark.sql("SELECT city, count(*) as count FROM v_patients GROUP BY city").show()
spark.sql("SELECT department, count(*) as count FROM v_appointments GROUP BY department").show()
spark.sql("SELECT department, avg(consultation_fee) as avg_fee FROM v_appointments GROUP BY department").show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|     NULL|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+

+-----------+-------+
| department|avg_fee|
+-----------+-------+
|  Neurology| 2000.0|
|Dermatology| 1000.0|
| Cardiology| 1500.0|
|Orthopedics| 2500.0|
+-----------+-------+



In [76]:
spark.sql("SELECT max(consultation_fee) as max_fee FROM v_appointments").show()
spark.sql("SELECT patient_id, count(*) as total_appointments FROM v_appointments GROUP BY patient_id").show()
spark.sql("""
    SELECT patient_id, sum(consultation_fee) as total_spent
    FROM v_appointments
    GROUP BY patient_id
    ORDER BY total_spent DESC LIMIT 5
""").show()

+-------+
|max_fee|
+-------+
|   2500|
+-------+

+----------+------------------+
|patient_id|total_appointments|
+----------+------------------+
|       108|                 1|
|       101|                 2|
|       103|                 1|
|       120|                 1|
|       107|                 1|
|       102|                 1|
|       105|                 1|
|       110|                 1|
|       104|                 1|
+----------+------------------+

+----------+-----------+
|patient_id|total_spent|
+----------+-----------+
|       101|       2500|
|       110|       2500|
|       104|       2500|
|       107|       2000|
|       102|       2000|
+----------+-----------+



In [77]:
print("\n--- Part 10: Integrated ETL Workflow Running ---")


--- Part 10: Integrated ETL Workflow Running ---


In [78]:
raw_p = spark.read.csv("data/patients.csv", header=True, inferSchema=True)
raw_a = spark.read.csv("data/appointments.csv", header=True, inferSchema=True)
raw_j = spark.read.option("multiline", "true").json("data/patient_preferences.json")

In [79]:
clean_p = raw_p.na.fill({"city": "Unknown", "blood_group": "Not Available"})
clean_a = raw_a.na.fill({"consultation_fee": 0})
clean_j = raw_j.select(
    F.col("patient_id"),
    F.col("preferred_hospital").alias("fav_hospital"),
    F.col("contact.phone").alias("phone_num"),
    F.col("contact.email").alias("email_addr")
).na.fill({"phone_num": "N/A", "email_addr": "N/A"})

In [80]:
unified_stage = clean_p.join(clean_a, "patient_id", "inner") \
                        .join(clean_j, "patient_id", "left")

In [81]:
final_reporting_df = unified_stage.withColumn("age_group",
    F.when(F.col("age") < 30, "Young")
     .when((F.col("age") >= 30) & (F.col("age") <= 50), "Middle-aged")
     .otherwise("Senior")
).withColumn("revenue_tier",
    F.when(F.col("consultation_fee") >= 2000, "High Premium")
     .when(F.col("consultation_fee") >= 1000, "Standard")
     .otherwise("Basic Value")
)

In [82]:
patient_spending_analytics = final_reporting_df.groupBy("patient_id", "patient_name").sum("consultation_fee")
dept_revenue_analytics = final_reporting_df.groupBy("department").sum("consultation_fee")

In [83]:
final_reporting_df.write.mode("overwrite").parquet("data/production_hospital_analytics.parquet")

In [84]:
print("==================================================================")
print("             FINAL HOSPITAL ANALYTICS PRODUCTION REPORT           ")
print("==================================================================")
final_reporting_df.select(
    "appointment_id", "patient_name", "city", "department", "consultation_fee", "revenue_tier"
).show(10, truncate=False)

             FINAL HOSPITAL ANALYTICS PRODUCTION REPORT           
+--------------+------------+---------+-----------+----------------+------------+
|appointment_id|patient_name|city     |department |consultation_fee|revenue_tier|
+--------------+------------+---------+-----------+----------------+------------+
|5001          |Rahul Sharma|Hyderabad|Cardiology |1500            |Standard    |
|5002          |Priya Reddy |Bangalore|Neurology  |2000            |High Premium|
|5003          |Rahul Sharma|Hyderabad|Dermatology|1000            |Standard    |
|5004          |Amit Kumar  |Mumbai   |Cardiology |1500            |Standard    |
|5005          |Sneha Patel |Chennai  |Orthopedics|2500            |High Premium|
|5006          |Farhan Ali  |Delhi    |Dermatology|1000            |Standard    |
|5007          |Arjun Verma |Pune     |Neurology  |2000            |High Premium|
|5008          |Nisha Reddy |Bangalore|Orthopedics|2500            |High Premium|
|5010          |Meera Nair  |Ko